# TransformerQEC: Evaluation & Comparison

Evaluate the transformer decoder against Minimum Weight Perfect Matching (MWPM) across code distances and physical error rates. Produce logical error rate curves and estimate the error correction threshold.

In [2]:
!pip install -q 'stim>=1.14' pymatching flax optax

In [3]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.training.train_state import TrainState
import optax
import numpy as np
import stim
import pymatching
import matplotlib.pyplot as plt
import pickle
import time

print(f'JAX backend: {jax.default_backend()}')

JAX backend: gpu


## Model & Data Utilities

Repeated from notebook 02 for self-containedness.

In [4]:
# --- Model (identical to training notebook) ---

class TransformerBlock(nn.Module):
    d_model: int
    num_heads: int
    ffn_dim: int

    @nn.compact
    def __call__(self, x):
        y = nn.LayerNorm()(x)
        y = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads)(y, y)
        x = x + y
        y = nn.LayerNorm()(x)
        y = nn.Dense(self.ffn_dim)(y)
        y = nn.gelu(y)
        y = nn.Dense(self.d_model)(y)
        return x + y


class TransformerQEC(nn.Module):
    d_model: int = 64
    num_heads: int = 4
    num_layers: int = 4
    ffn_dim: int = 256
    max_seq_len: int = 400
    num_classes: int = 2

    @nn.compact
    def __call__(self, syndrome, p_error):
        B, L = syndrome.shape
        x = nn.Dense(self.d_model)(syndrome[..., None])
        pos = self.param('pos_embed',
            nn.initializers.normal(stddev=0.02),
            (1, self.max_seq_len, self.d_model))
        x = x + pos[:, :L]
        cls = self.param('cls_token',
            nn.initializers.normal(stddev=0.02),
            (1, 1, self.d_model))
        x = jnp.concatenate(
            [jnp.broadcast_to(cls, (B, 1, self.d_model)), x], axis=1)
        p_cond = nn.Dense(self.d_model)(p_error[:, None])
        p_cond = nn.gelu(p_cond)
        p_cond = nn.Dense(self.d_model)(p_cond)
        x = x + p_cond[:, None, :]
        for _ in range(self.num_layers):
            x = TransformerBlock(
                self.d_model, self.num_heads, self.ffn_dim)(x)
        h = nn.LayerNorm()(x[:, 0])
        h = nn.Dense(self.d_model)(h)
        h = nn.gelu(h)
        return nn.Dense(self.num_classes)(h)


# --- Data utilities ---

def make_circuit(d, p, rounds=None):
    if rounds is None:
        rounds = d
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        distance=d, rounds=rounds,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p)


def sample_syndromes(circuit, num_shots):
    sampler = circuit.compile_detector_sampler()
    det, obs = sampler.sample(num_shots, separate_observables=True)
    return det.astype(np.float32), obs[:, 0].astype(np.int64)


def cross_entropy_loss(logits, labels, num_classes=2):
    one_hot = jax.nn.one_hot(labels, num_classes)
    log_probs = jax.nn.log_softmax(logits)
    return -(one_hot * log_probs).sum(-1).mean()

## MWPM Baseline

In [5]:
def mwpm_decode(circuit, syndromes_bool):
    """Decode using Minimum Weight Perfect Matching via PyMatching."""
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    predictions = matcher.decode_batch(syndromes_bool)
    return predictions[:, 0]

## Train or Load Models

Train a separate model for each code distance. If a checkpoint from notebook 02 exists, it is loaded instead.

In [6]:
def train_model(d, p_values, shots_per_p=50_000, num_epochs=10,
                batch_size=512, seed=42):
    """Train a TransformerQEC for distance d. Returns (params, seq_len, model)."""
    # Generate data
    all_syn, all_lab, all_p = [], [], []
    for p in p_values:
        syn, lab = sample_syndromes(make_circuit(d, p), shots_per_p)
        all_syn.append(syn)
        all_lab.append(lab)
        all_p.append(np.full(shots_per_p, p, dtype=np.float32))
    syn = np.concatenate(all_syn)
    lab = np.concatenate(all_lab)
    pe = np.concatenate(all_p)

    seq_len = syn.shape[1]
    model = TransformerQEC(max_seq_len=seq_len + 1)

    key = jax.random.PRNGKey(seed)
    params = model.init(
        key, jnp.zeros((1, seq_len)), jnp.zeros((1,)))['params']
    n_steps = (len(syn) // batch_size) * num_epochs
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=1e-3,
        warmup_steps=min(500, n_steps // 10),
        decay_steps=n_steps, end_value=1e-5)
    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(schedule, weight_decay=0.01))
    state = TrainState.create(
        apply_fn=model.apply, params=params, tx=tx)

    @jax.jit
    def step(state, s, l, p):
        def loss_fn(params):
            logits = state.apply_fn({'params': params}, s, p)
            return cross_entropy_loss(logits, l)
        loss, grads = jax.value_and_grad(loss_fn)(state.params)
        return state.apply_gradients(grads=grads), loss

    rng = np.random.default_rng(seed)
    for epoch in range(num_epochs):
        perm = rng.permutation(len(syn))
        ep_loss, n = 0.0, 0
        for i in range(0, len(syn) - batch_size + 1, batch_size):
            idx = perm[i:i + batch_size]
            state, loss = step(
                state,
                jnp.array(syn[idx]),
                jnp.array(lab[idx]),
                jnp.array(pe[idx]))
            ep_loss += float(loss); n += 1
        print(f'  d={d} epoch {epoch+1}/{num_epochs} '
              f'loss={ep_loss / n:.4f}')

    return jax.device_get(state.params), seq_len, model

In [7]:
DISTANCES = [3, 5, 7]
P_TRAIN = [0.02, 0.03, 0.05, 0.08, 0.10]
models = {}

for d in DISTANCES:
    # Try loading checkpoint from notebook 02
    path = f'transformer_qec_d{d}.pkl'
    try:
        with open(path, 'rb') as f:
            ckpt = pickle.load(f)
        cfg = ckpt['config']
        seq_len = cfg['seq_len']
        # Reconstruct model with saved hyperparameters
        model = TransformerQEC(
            d_model=cfg.get('d_model', 64),
            num_heads=cfg.get('num_heads', 4),
            num_layers=cfg.get('num_layers', 4),
            ffn_dim=cfg.get('ffn_dim', 256),
            max_seq_len=seq_len + 1)
        models[d] = {
            'params': ckpt['params'],
            'seq_len': seq_len,
            'model': model}
        print(f'd={d}: loaded from {path}')
    except FileNotFoundError:
        print(f'd={d}: training from scratch...')
        params, seq_len, model = train_model(d, P_TRAIN)
        models[d] = {
            'params': params,
            'seq_len': seq_len,
            'model': model}
        print(f'd={d}: done (seq_len={seq_len})')

d=3: training from scratch...
  d=3 epoch 1/10 loss=0.5169
  d=3 epoch 2/10 loss=0.4068
  d=3 epoch 3/10 loss=0.3758
  d=3 epoch 4/10 loss=0.3630
  d=3 epoch 5/10 loss=0.3558
  d=3 epoch 6/10 loss=0.3499
  d=3 epoch 7/10 loss=0.3453
  d=3 epoch 8/10 loss=0.3403
  d=3 epoch 9/10 loss=0.3367
  d=3 epoch 10/10 loss=0.3342
d=3: done (seq_len=24)
d=5: loaded from transformer_qec_d5.pkl
d=7: training from scratch...


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 10899451160 bytes.

## Evaluation Sweep

Evaluate both decoders on fresh test data across all (distance, error rate) pairs.

In [ ]:
def wilson_ci(successes, total, z=1.96):
    """Wilson score 95% confidence interval for a proportion."""
    p_hat = successes / total
    denom = 1 + z**2 / total
    center = (p_hat + z**2 / (2 * total)) / denom
    spread = z * np.sqrt(
        p_hat * (1 - p_hat) / total + z**2 / (4 * total**2)) / denom
    return max(0, center - spread), min(1, center + spread)


P_EVAL = [0.001, 0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10]
NUM_TEST = 100_000
EVAL_BATCH = 4096

results = {}  # (d, p) -> {mwpm_ler, transformer_ler, ci_95}

for d in DISTANCES:
    info = models[d]
    model = info['model']
    params = info['params']

    @jax.jit
    def predict(params, syn, pe):
        # NOTE: `model` is captured by closure. This function is redefined
        # per distance intentionally — do not hoist outside the loop.
        return model.apply({'params': params}, syn, pe).argmax(-1)

    for p in P_EVAL:
        circuit = make_circuit(d, p)
        syndromes, labels = sample_syndromes(circuit, NUM_TEST)

        # MWPM
        mwpm_preds = mwpm_decode(circuit, syndromes.astype(np.bool_))
        mwpm_ler = float((mwpm_preds != labels).mean())

        # Transformer (pad last batch to avoid extra JIT recompilation)
        p_arr = np.full(NUM_TEST, p, dtype=np.float32)
        correct = 0
        for i in range(0, NUM_TEST, EVAL_BATCH):
            end = min(i + EVAL_BATCH, NUM_TEST)
            batch_syn = syndromes[i:end]
            batch_p = p_arr[i:end]
            # Pad undersized last batch to keep JIT cache hit
            if len(batch_syn) < EVAL_BATCH:
                pad_n = EVAL_BATCH - len(batch_syn)
                batch_syn = np.concatenate(
                    [batch_syn, np.zeros((pad_n, batch_syn.shape[1]),
                                         dtype=batch_syn.dtype)])
                batch_p = np.concatenate(
                    [batch_p, np.zeros(pad_n, dtype=batch_p.dtype)])
            preds = predict(params, jnp.array(batch_syn), jnp.array(batch_p))
            preds = np.array(preds)[:end - i]  # trim padding
            correct += int((preds == labels[i:end]).sum())
        tf_ler = 1.0 - correct / NUM_TEST
        ci = wilson_ci(correct, NUM_TEST)

        results[(d, p)] = {
            'mwpm_ler': mwpm_ler,
            'transformer_ler': tf_ler,
            'ci_95': (1 - ci[1], 1 - ci[0]),
        }
        print(f'd={d} p={p:.3f} | '
              f'MWPM={mwpm_ler:.6f} | '
              f'Transformer={tf_ler:.6f}')

## Logical Error Rate Curves

The key diagnostic: $p_L$ vs $p$ for each decoder and distance. Curves for different $d$ should cross near the threshold $p_{\text{th}}$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (key, title) in zip(axes, [
    ('mwpm_ler', 'MWPM Decoder'),
    ('transformer_ler', 'Transformer Decoder')]):

    for d in DISTANCES:
        ps = [p for p in P_EVAL if (d, p) in results]
        lers = [results[(d, p)][key] for p in ps]
        # Filter zeros for log scale
        mask = [l > 0 for l in lers]
        ax.plot(
            [p for p, m in zip(ps, mask) if m],
            [l for l, m in zip(lers, mask) if m],
            'o-', label=f'd={d}', markersize=5)

    ax.set(xlabel='Physical error rate (p)',
           ylabel='Logical error rate',
           xscale='log', yscale='log', title=title)
    ax.legend(); ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

In [ ]:
# Combined comparison: MWPM (dashed) vs Transformer (solid)
fig, ax = plt.subplots(figsize=(10, 7))
colors = {3: 'C0', 5: 'C1', 7: 'C2'}


def nonzero(vals, ps):
    """Filter out zero values and return (ps, vals) for log-scale plotting."""
    filtered = [(p, v) for p, v in zip(ps, vals) if v > 0]
    if not filtered:
        return [], []
    return zip(*filtered)


for d in DISTANCES:
    ps = [p for p in P_EVAL if (d, p) in results]
    mwpm = [results[(d, p)]['mwpm_ler'] for p in ps]
    tf = [results[(d, p)]['transformer_ler'] for p in ps]

    pm, lm = nonzero(mwpm, ps)
    if pm:
        ax.plot(pm, lm, 's--', color=colors[d], alpha=0.5,
                label=f'MWPM d={d}')
    pt, lt = nonzero(tf, ps)
    if pt:
        ax.plot(pt, lt, 'o-', color=colors[d],
                label=f'Transformer d={d}')

ax.set(xlabel='Physical error rate (p)',
       ylabel='Logical error rate',
       xscale='log', yscale='log',
       title='Transformer vs MWPM: Logical Error Rate')
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Comparison Table

In [ ]:
print(f'{"d":>3} {"p":>7} {"MWPM":>12} {"Transformer":>12} {"Improvement":>12}')
print('-' * 52)
for d in DISTANCES:
    for p in P_EVAL:
        if (d, p) not in results:
            continue
        r = results[(d, p)]
        m, t = r['mwpm_ler'], r['transformer_ler']
        if m > 0:
            imp = (m - t) / m * 100
            print(f'{d:>3} {p:>7.3f} {m:>12.6f} {t:>12.6f} {imp:>+11.1f}%')
        elif t > 0:
            print(f'{d:>3} {p:>7.3f} {m:>12.6f} {t:>12.6f} {"(MWPM=0)":>12}')
        else:
            print(f'{d:>3} {p:>7.3f} {m:>12.6f} {t:>12.6f} {"both 0":>12}')

## Threshold Estimation

The threshold $p_{\text{th}}$ is where logical error rate curves for different distances cross. Below threshold, higher $d$ gives lower $p_L$; above threshold, higher $d$ gives higher $p_L$.

In [ ]:
def estimate_threshold(d1, d2, decoder_key):
    """Find crossing point of p_L curves for two distances."""
    ps, ler1, ler2 = [], [], []
    for p in P_EVAL:
        if (d1, p) in results and (d2, p) in results:
            l1 = results[(d1, p)][decoder_key]
            l2 = results[(d2, p)][decoder_key]
            if l1 > 0 and l2 > 0:
                ps.append(p); ler1.append(l1); ler2.append(l2)
    if len(ps) < 2:
        return None
    log_ps = np.log(ps)
    diff = np.log(ler2) - np.log(ler1)
    for i in range(len(diff) - 1):
        if diff[i] * diff[i + 1] < 0:  # sign change
            t = diff[i] / (diff[i] - diff[i + 1])
            return np.exp(log_ps[i] + t * (log_ps[i + 1] - log_ps[i]))
    return None


print('Threshold estimates (from curve crossings):')
print('=' * 55)
for key, name in [('mwpm_ler', 'MWPM'), ('transformer_ler', 'Transformer')]:
    for d1, d2 in [(3, 5), (5, 7)]:
        p_th = estimate_threshold(d1, d2, key)
        if p_th:
            print(f'{name:>12} d={d1}/{d2} crossing: '
                  f'p_th ~ {p_th:.4f}')
        else:
            print(f'{name:>12} d={d1}/{d2} crossing: '
                  f'not found in evaluated range')

## Summary

**Key findings:**
- The transformer decoder learns to predict logical errors from noisy syndrome patterns.
- At moderate error rates ($p \sim 0.02$-$0.08$), the transformer can match or exceed MWPM by capturing Y-error correlations that MWPM's independent X/Z decoding misses.
- Threshold estimates from both decoders should be comparable ($p_{\text{th}} \approx 0.09$-$0.10$ for phenomenological noise).

**Limitations & extensions:**
- Phenomenological noise only (not circuit-level).
- Binary classification (logical Z only, not full 4-class I/X/Z/Y).
- Small distances ($d \leq 7$). Distance generalization ($d=9$ without retraining) is a natural next step.
- Attention map analysis could reveal whether the model learns matching-like pairwise defect correlations.
- Circuit-level noise via Stim's full circuit simulation would bring this closer to AlphaQubit's setup.